# MNIST FPGA Implementation Cross-Comparison

This notebook compares the major MNIST FPGA implementations developed during the project.

The implementations are intentionally separated by purpose:

1. **Local Live Classification**  
   Camera-derived image classified locally on the FPGA.

2. **Local Formal Measurement**  
   Controlled MNIST inputs used to measure local FPGA accuracy, latency, throughput, utilization, and power.

3. **Cloud Formal Measurement**  
   Controlled MNIST inputs transferred through the Zynq Processing System to Google Cloud Run for formal accuracy, latency, throughput, power, and memory characterization.

4. **Cloud Live Classification**  
   Frozen camera-derived image transferred to Cloud Run and the returned prediction displayed on the ZedBoard LEDs.

The live implementations demonstrate end-to-end functionality. The formal measurement implementations provide the quantitative benchmark results.


## 1. Common MNIST Model

The MNIST reference network used throughout this work is a fully connected feed-forward network:

```text
784 → 30 → 20 → 10
```

with sigmoid activations and a final class selected from the ten output values.

Reference software accuracy:

```text
9663 / 10000
96.63%
```

The FPGA-local implementation uses the generated ZyNet fixed-point RTL representation of this trained network.

The Cloud Run implementation executes the floating-point software network.


## 2. Implementation Progression

```text
trained MNIST model
        ↓
local FPGA live classification
        ↓
local FPGA controlled measurement
        ↓
FPGA-to-Cloud controlled measurement
        ↓
FPGA-to-Cloud live camera classification
```

Each stage served a different experimental purpose rather than replacing the previous stage.


## 3. High-Level Architecture Comparison

| Implementation | Input | Inference Location | Result Path | Primary Purpose |
|---|---|---|---|---|
| Local Live | Frozen camera-derived 28×28 image | FPGA programmable logic | Local result → LEDs | Functional live demonstration |
| Local Measurement | Controlled MNIST test images | FPGA programmable logic | Local result/counters | Formal local benchmarking |
| Cloud Measurement | Controlled MNIST test images | Google Cloud Run | PS → mailbox → PL | Formal cloud-assisted benchmarking |
| Cloud Live | Frozen camera-derived 28×28 image | Google Cloud Run | PS → mailbox → LEDs | Functional live cloud demonstration |


## 4. Accuracy Comparison

Formal accuracy was measured only with controlled dataset inputs.

| Configuration | Correct | Tested | Accuracy |
|---|---:|---:|---:|
| Software reference model | 9,663 | 10,000 | 96.63% |
| FPGA local formal measurement | 969 | 1,000 | 96.90% |
| FPGA-to-Cloud formal measurement | 970 | 1,000 | 97.00% |

The two FPGA experimental paths produced very similar accuracy.

The local FPGA result reflects the fixed-point ZyNet implementation, while the cloud result reflects the floating-point software model executed remotely.

The live camera implementations were used to demonstrate end-to-end classification and were not used to calculate formal dataset accuracy.


In [ ]:
# Accuracy comparison
accuracy = {
    "Software reference": 96.63,
    "FPGA local": 96.90,
    "FPGA cloud": 97.00,
}

for name, value in accuracy.items():
    print(f"{name:20s}: {value:.2f}%")

print()
print(f"Cloud - local difference: {accuracy['FPGA cloud'] - accuracy['FPGA local']:.2f} percentage points")
print(f"Local - reference difference: {accuracy['FPGA local'] - accuracy['Software reference']:.2f} percentage points")


## 5. Latency Comparison

### Local FPGA

Formal local inference latency:

```text
1,646 cycles
25 MHz
40 ns per cycle
65.84 µs per image
```

### FPGA-to-Cloud

Formal cloud-assisted latency:

```text
Typical steady-state mean: approximately 58–60 ms
Minimum observed: 51.934 ms
Maximum observed: 159.652 ms
```

These are different measurement paths:

- **Local FPGA latency** measures the local controlled inference path.
- **Cloud latency** includes client request transmission, network delay, Cloud Run execution, and response receipt.

The live cloud implementation may print diagnostic request latency, but those values are not part of the formal 100-image timing dataset.


In [ ]:
local_latency_us = 65.84
cloud_typical_ms = 59.0

cloud_typical_us = cloud_typical_ms * 1000
ratio = cloud_typical_us / local_latency_us

print(f"Local FPGA latency: {local_latency_us:.2f} µs")
print(f"Representative cloud latency: {cloud_typical_ms:.1f} ms")
print(f"Representative cloud/local latency ratio: {ratio:,.1f}×")


## 6. Throughput Comparison

### Local FPGA

For 100 controlled images:

```text
164,899 cycles
6.59596 ms total
299 cycles of controller/inter-image overhead
15,160.8 classifications/s
```

### FPGA-to-Cloud

Typical formal steady-state throughput:

```text
approximately 16.8–17.1 classifications/s
```

The local FPGA therefore provides orders-of-magnitude greater classification throughput because inference executes directly in programmable logic without network communication.


In [ ]:
local_throughput = 15160.8
cloud_throughput = 16.9

print(f"Local FPGA throughput: {local_throughput:,.1f} classifications/s")
print(f"Representative cloud throughput: {cloud_throughput:.1f} classifications/s")
print(f"Local/cloud throughput ratio: {local_throughput/cloud_throughput:,.1f}×")


## 7. Local FPGA Resource Baseline

The clean local FPGA implementation used for the primary local resource comparison was:

| Resource | Used | Device Total | Utilization |
|---|---:|---:|---:|
| LUTs | 8,455 | 53,200 | 15.89% |
| Registers | 4,075 | 106,400 | 3.83% |
| BRAM tiles | 20 | 140 | 14.29% |
| DSPs | 0 | 220 | 0.00% |

This is the preferred local deployment baseline.

Measurement-specific builds used additional instrumentation and should not replace this clean deployment footprint.


## 8. Local Measurement-Build Overhead

### Accuracy build

```text
LUTs:       8,626
Registers:  4,088
BRAM:       39.5
DSPs:       1
```

### Speed build

```text
LUTs:      10,416
Registers:  7,271
BRAM:          46
DSPs:           1
```

The added ROM, counters, and timing/debug instrumentation explain why these builds are larger than the clean local implementation.


## 9. Cloud Measurement FPGA Resources

### Formal cloud measurement build

| Resource | Used | Utilization |
|---|---:|---:|
| LUTs | 4,043 | 7.60% |
| Registers | 5,511 | 5.18% |
| BRAM tiles | 36 | 25.71% |
| DSPs | 2 | 0.91% |

This build included the controlled test workload and measurement instrumentation.

### Reduced cloud power/memory build

| Resource | Used | Utilization |
|---|---:|---:|
| LUTs | 2,181 | 4.10% |
| Registers | 2,254 | 2.12% |
| BRAM tiles | 29.5 | 21.07% |
| DSPs | 1 | 0.45% |

The local neural-network hardware was removed before cloud power and memory characterization because local inference was not required in the cloud-assisted architecture.


## 10. Deployment Resource Comparison

For the most meaningful architectural comparison:

| Resource | Local FPGA Baseline | Reduced Cloud-Assisted FPGA |
|---|---:|---:|
| LUTs | 8,455 | 2,181 |
| Registers | 4,075 | 2,254 |
| BRAM tiles | 20 | 29.5 |
| DSPs | 0 | 1 |

Interpretation:

- Local inference requires substantially more programmable-logic computation.
- The reduced cloud-assisted implementation uses fewer LUTs and registers because the neural-network calculation is performed remotely.
- The cloud architecture still requires PL/PS communication and buffering infrastructure.
- BRAM usage should therefore not be interpreted as neural-network storage alone.


In [ ]:
local = {"LUT": 8455, "Registers": 4075, "BRAM": 20.0, "DSP": 0}
cloud = {"LUT": 2181, "Registers": 2254, "BRAM": 29.5, "DSP": 1}

for key in local:
    diff = cloud[key] - local[key]
    print(f"{key:10s}: local={local[key]:7.1f}  cloud={cloud[key]:7.1f}  cloud-local={diff:+7.1f}")


## 11. Vivado Power Estimates

### Local FPGA clean baseline

```text
Total on-chip power: 1.757 W
Dynamic power:       1.608 W
Static power:        0.149 W
```

### Cloud reduced power build

```text
Total on-chip power: 1.771 W
Dynamic power:       1.621 W
Static power:        0.150 W
```

### Cloud live build

```text
Total on-chip power: 1.769 W
Dynamic power:       1.621 W
Static power:        0.148 W
```

Vivado estimates and physical meter observations represent different measurement boundaries and should not be treated as interchangeable.


## 12. Physical Power Observations

### Local FPGA

Observed board/system power was approximately:

```text
unprogrammed:             2.6–2.7 W
programmed/local states:  2.7–2.8 W
continuous inference:     2.7–2.8 W
occasional observation:   2.9 W
```

The external meter did not clearly resolve a sustained neural-network-only increment.

### Cloud-assisted FPGA

The cloud branches include physical Kill A Watt photographs and videos documenting the programmed, networked, SDK, and active cloud-inference states.

These measurements include the local ZedBoard/PS/network system but **do not include the electrical power consumed by Google Cloud Run infrastructure**.

For this reason, physical local and cloud-assisted power values should be interpreted as platform-boundary measurements rather than complete end-to-end energy cost.


## 13. Memory Comparison

### Local FPGA

The local implementation stores the neural-network parameters and intermediate logic in FPGA resources.

Primary hardware resource categories include:

```text
LUTs
registers
BRAM
fixed-point parameter memories
```

### Cloud-assisted FPGA

The remote model is not stored as an FPGA inference engine.

Instead, the Zynq PS requires software/network memory, including approximately:

```text
TLS receive buffer        32 KB
Mbed TLS memory pool     128 KB
HTTP response buffer       4 KB
HTTP header buffer       ~512 B
784-byte image buffer
```

plus lwIP, DHCP, DNS, TCP, TLS structures, heap, stack, and application state.

Therefore, FPGA BRAM counts and PS software-memory consumption measure different types of memory use.


## 14. Live Local vs. Live Cloud

### Local Live

```text
camera
  ↓
freeze image
  ↓
local ZyNet inference
  ↓
prediction
  ↓
LED output
```

### Cloud Live

```text
camera
  ↓
freeze image
  ↓
mailbox
  ↓
Zynq PS
  ↓
Cloud Run
  ↓
returned prediction
  ↓
LED output
```

Both live versions use a comparable physical input/output workflow, but the inference location changes from programmable logic to the remote cloud service.


## 15. What Each Implementation Demonstrates

| Implementation | Demonstrates |
|---|---|
| Local Live | Camera preprocessing, frozen-image handling, on-FPGA inference, physical output |
| Local Measurement | Controlled local accuracy, deterministic latency, high throughput, FPGA resource cost |
| Cloud Measurement | Controlled cloud accuracy, networked latency/throughput, PS/TLS cost, reduced PL resource requirement |
| Cloud Live | End-to-end camera → FPGA → PS → Cloud Run → FPGA → LED operation |


## 16. Key Cross-Platform Findings

### Accuracy

The local fixed-point FPGA implementation and cloud floating-point implementation both remained close to the 96.63% software reference result.

### Performance

Local FPGA inference is dramatically faster than cloud-assisted inference because it eliminates network transport and remote-service latency.

### Resource placement

Local inference consumes more programmable-logic computation, while the cloud-assisted design moves the neural-network execution away from the FPGA and instead requires PS networking, TLS, and buffering resources.

### Power boundary

Local physical measurements capture the complete local inference system. Cloud-assisted physical measurements capture only the local client side and exclude remote data-center energy.

### Functional deployment

Both live implementations successfully use the same basic camera-derived 28 × 28 input workflow and provide a physical prediction output on the ZedBoard.


## 17. Summary Table

| Metric | Local FPGA | FPGA-to-Cloud |
|---|---:|---:|
| Formal accuracy | 96.90% | 97.00% |
| Typical latency | 65.84 µs | ~58–60 ms |
| Typical throughput | 15,160.8/s | ~16.8–17.1/s |
| Deployment LUTs | 8,455 | 2,181 |
| Deployment registers | 4,075 | 2,254 |
| Deployment BRAM | 20 | 29.5 |
| Deployment DSPs | 0 | 1 |
| Vivado power estimate | 1.757 W | 1.771 W |
| Live camera path | Yes | Yes |
| Prediction on LEDs | Yes | Yes |
| Remote network dependency | No | Yes |

The table deliberately uses the clean local implementation and reduced cloud-assisted implementation for resource comparison rather than the larger instrumented measurement builds.


## 18. Repository Branch Mapping

The comparison is supported by the following branch groups:

```text
MNIST-FPGA-Live-Classification-Code
MNIST-FPGA-Live-Classification-Notebook-and-Data

MNIST-FPGA-Local-Classification-Measurement-Code
MNIST-FPGA-Local-Classification-Measurement-Notebook-and-Data

MNIST-FPGA-Cloud-Classification-Measurement-Code
MNIST-FPGA-Cloud-Classification-Measurement-Notebook-and-Data

MNIST-FPGA-Cloud-Live-Classification-Code
MNIST-FPGA-Cloud-Live-Classification-Notebook-and-Data

Cloud
    Cloud Run service setup and service-side evidence
```

This cross-comparison notebook should sit at a level where it can reference all of those implementation-specific branches without duplicating their full development records.


## 19. Scope

This notebook compares the MNIST FPGA implementations and results only.

It should not be used to combine measurement boundaries that were intentionally different:

- live functional demonstrations are not formal accuracy datasets;
- Cloud Run server-side latency is not client-side FPGA cloud latency;
- measurement-build utilization is not the preferred deployment resource footprint;
- Vivado power estimates are not identical to physical meter measurements; and
- cloud-client power excludes remote cloud infrastructure.

Keeping these boundaries explicit allows the implementations to be compared without overstating equivalence between local and cloud execution.
